In [ ]:
import numpy as np
import pandas as pd
import os,sys
import json, h5py

from openai import OpenAI
# Read the key from file
with open("openai_key.txt", "r") as f:
    openai_key = f.read().strip()

# Set environment variable for OpenAI client
os.environ["OPENAI_API_KEY"] = openai_key

In [ ]:
input_dir = "combined_UCE_56M"
meta_file = "obs.tsv"
uce_file = "uce.h5"
chunk_size = 1000

In [ ]:
meta = pd.read_csv(os.path.join(input_dir, meta_file), sep="\t")

## assign cell category based on cell type label

In [ ]:
# get cell_type summary information, export to file
celltype_counts = meta["cell_type"].value_counts().reset_index()
celltype_counts.columns = ["cell_type", "count"]
celltype_counts.to_csv(os.path.join(input_dir, "celltype_counts.tsv"), sep="\t", index=False)
celltype_counts

In [ ]:
# load cell_type_summary information back
# celltype_counts = pd.read_csv(os.path.join(input_dir,"celltype_counts.tsv"), sep="\t")

In [ ]:
cell_types = celltype_counts["cell_type"].tolist()

### Python code to call the OpenAI API and assign each cell type to one of the four categories:

brain_nervous_system

immune

malignant

other

The code uses the Responses API (the current recommended OpenAI API interface).

In [ ]:
system_prompt = """
You are a cell type annotation assistant. 
Your task is to assign each cell type to exactly one of the following high-level categories:

- malignant
- brain_nervous_system
- immune
- epithelial
- stromal
- endothelial
- other
- unknown

Classification rules:

Step 0: Determine if the cell type is explicitly unknown.
- If the label contains words like "unknown", "unclassified", "undetermined", or "NA", assign "unknown".
- This rule has the highest priority and overrides all other rules.

Step 1: Determine if the cell type is tumor/malignant.
- If the cell type indicates tumor, cancer, carcinoma, glioblastoma, glioma, transformed, blastoma, or any malignant properties → assign "malignant".
- This rule has top priority after unknown. Even if the cell type appears neuron-like, glia-like, or immune-like, classify it as "malignant".

Step 2: Determine if the cell type contributes to the nervous system.
- If the cell is exclusively in the brain or nervous system of any species, assign "brain_nervous_system".
- Include normal developmental precursors that give rise to neurons or glia, e.g.:
    - glioblast → astrocytes or oligodendrocytes
    - neuroepithelial stem cells
    - neuroplacodal cells → sensory neurons
    - neural crest derivatives that produce peripheral neurons or glia
- Do not assign brain_nervous_system to ubiquitous or multi-system cells such as pericytes, endothelial cells, fibroblasts, except when they are explicitly tissue-specific to the nervous system (e.g., choroid plexus–specific cells, meninges-specific cells, or other CNS-restricted supporting cell types).

Step 3: For all other non-malignant cells not assigned to brain_nervous_system:
- immune: T cell, B cell, NK cell, macrophage, monocyte, dendritic cell, granulocyte, etc.
- epithelial: cells forming epithelial layers such as keratinocytes, hepatocytes, epithelial cells from organ linings, ductal cells, etc.
- stromal: connective tissue, fibroblasts, mesenchymal cells, pericytes, and other supportive tissue cells.
- endothelial: blood vessel or lymphatic endothelial cells, including vascular endothelial cells and lymphatic endothelial cells.
- other: any ambiguous or unrelated cell type.

Additional instructions:
- Use biological knowledge from single-cell RNA-seq studies.
- Return ONLY a JSON object with this structure:

{
  "annotations": [
    {"cell_type": "...", "category": "..."},
    ...
  ]
}

Each cell type must be assigned exactly one category.

"""

In [ ]:
# user message
user_message = {
    "cell_types": cell_types
}

In [ ]:
client = OpenAI()

response = client.responses.create(
    model="gpt-4.1", # "gpt-4o"
    input=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": json.dumps(user_message)}
    ]
)

# Extract JSON from the response
content = response.output_text
annotations = json.loads(content)["annotations"]

# Convert back to DataFrame
df_annotations = pd.DataFrame(annotations)

print(df_annotations)

In [ ]:
# Merge counts from original file
df_annotations = df_annotations.merge(celltype_counts, on="cell_type", how="left")

In [ ]:
# Define the custom order
category_order = [
    "malignant",
    "brain_nervous_system",
    "immune",
    "epithelial",
    "stromal",
    "endothelial",
    "other",
    "unknown"
]

df_annotations['category'] = pd.Categorical( 
    df_annotations['category'], 
    categories=category_order, 
    ordered=True )

# Sort the DataFrame by this custom order
df_sorted = df_annotations.sort_values(
    by=['category', 'count'],  # first category, then count
    ascending=[True, False]    # category ascending, counts descending
)

df_sorted

In [ ]:
# Save final TSV
df_sorted.to_csv(os.path.join(input_dir, "celltype_category_assignments.tsv"),
                      sep="\t", index=False)

print("Saved: celltype_category_assignments.tsv")

## create row indices for neuro subset

In [ ]:
# read category info back from file
df_annotations = pd.read_csv(
    os.path.join(input_dir, "celltype_category_assignments.tsv"),
    sep="\t"
)

In [ ]:
# Step 1: Get the cell types in the 'brain_nervous_system' category
brain_cell_types = df_annotations.loc[
    df_annotations['category'] == 'brain_nervous_system', 'cell_type'
].tolist()

# Step 2 — get **row positions**, not index labels
row_indices = np.where(meta['cell_type'].isin(brain_cell_types))[0]

row_indices

In [ ]:
len(row_indices)

## row indices chunked shuffle

In [ ]:
row_indices = np.asarray(row_indices)
N = len(row_indices)

# number of full chunks
num_full_chunks = N // chunk_size

# last chunk start index
last_chunk_start = num_full_chunks * chunk_size

# Build array of chunk IDs for full chunks only
chunk_ids = np.arange(num_full_chunks)

# Shuffle only these
rng = np.random.default_rng()
rng.shuffle(chunk_ids)

# Reassemble
shuffled_indices = np.concatenate([
    # shuffled full chunks
    row_indices[cid*chunk_size : (cid+1)*chunk_size]
    for cid in chunk_ids
] + [
    # append the last incomplete chunk unchanged
    row_indices[last_chunk_start:]
])


out_file = os.path.join(input_dir, "shuffled_indices_20M_neuro.npy")
# Save
np.save(out_file, shuffled_indices)

## Extract metadata using those indices

In [ ]:
output_dir = "combined_UCE_20M_Brain_NervousSystem_shuffled"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Load shuffled indice back later
out_file = os.path.join(input_dir, "shuffled_indices_20M_neuro.npy")
shuffled_indices = np.load(out_file)

In [ ]:
meta_subset = meta.iloc[shuffled_indices]

In [ ]:
meta_subset.to_csv(os.path.join(output_dir, "obs.tsv"), sep="\t", index=False)

## Extract uce.h5 using those indices

In [ ]:
# handle to uce.h5 
fuce = h5py.File(os.path.join(input_dir, uce_file), "r")
uce = fuce["data"] # h5 handle
print(uce.shape)
#fuce.close()

In [ ]:
def smart_h5py_batch_read(dataset, sorted_indices):
    """Efficiently read sorted indices from HDF5 by grouping nearby rows."""
    output = []
    is_sorted = np.all(sorted_indices[:-1] <= sorted_indices[1:])
    assert(is_sorted)
    #sorted_indices = np.sort(np.array(sorted_indices, dtype=int))  # Explicit sort
    i = 0
    while i < len(sorted_indices):
        start = sorted_indices[i]
        j = i
        while (j + 1 < len(sorted_indices) and sorted_indices[j + 1] == sorted_indices[j] + 1):
            j += 1
        end = sorted_indices[j]
        chunk = dataset[start:end+1, :] # Read slice
        output.append(chunk)
        i = j + 1
    return np.vstack(output)

In [ ]:
output_h5 = os.path.join(output_dir, uce_file)
batch_size = chunk_size
total_len = len(shuffled_indices)

with h5py.File(output_h5, "w") as f:
    start_idx = 0
    dset = None

    for i in range(0, total_len, batch_size):
        if i % 20_000 == 0:
            print("Processing cell", start_idx)
        idx_batch_sorted = shuffled_indices[i:i+batch_size]
        batch = smart_h5py_batch_read(uce, idx_batch_sorted)  # shape: (batch_rows, 1280)

        if dset is None:
            # Create the dataset dynamically using shape of first batch
            n_cols = batch.shape[1]
            dset = f.create_dataset(
                "data", shape=(total_len, n_cols), dtype=batch.dtype
            )

        end_idx = start_idx + batch.shape[0]
        dset[start_idx:end_idx, :] = batch
        start_idx = end_idx

## assign cell category based on publication and cell type label

In [ ]:
# get cell_type summary information, export to file
#meta["collection_doi"] = "10.1038/s41593-024-01774-5"
cell_entries_counts = meta[["cell_type","collection_doi"]].value_counts().reset_index()
cell_entries_counts.columns = ["cell_type", "collection_doi", "count"]
cell_entries_counts.to_csv(os.path.join(input_dir, "cell_entries_counts.tsv"), sep="\t", index=False)

In [ ]:
cell_entries_counts

In [ ]:
df_summary = pd.read_csv(os.path.join(input_dir, "cell_entries_counts.tsv"), sep="\t")

# Convert to list of dicts
cell_entries = df_summary[["cell_type", "collection_doi"]].to_dict(orient="records")
cell_entries

In [ ]:
system_prompt = """
You are a cell type annotation assistant. 
You are given a cell type label and a publication DOI. 

Your task is to assign each cell type to exactly one of the following categories:

- malignant
- brain_nervous_system
- immune
- other

Classification rules:

Step 1: Check the publication (via DOI):
- If the publication is specifically about the brain or nervous system, assign "brain_nervous_system" to all non-malignant cells, regardless of whether the cell is immune-like or not. 
- If the publication is about multiple organs, whole body, or is ambiguous, skip this step and use cell type information instead.
- This assignment is overridden by tumor/malignant classification.

Step 2: Check if the cell type is tumor/malignant:
- If the cell type indicates tumor, cancer, carcinoma, glioma, transformed, blastoma, or any malignant properties → assign "malignant".
- Tumor classification **overrides any previous assignment from publication**.

Step 3: For remaining cells not assigned yet (non-brain, non-malignant):
- immune: T cell, B cell, NK cell, macrophage, monocyte, dendritic cell, granulocyte, etc.
- other: any ambiguous or unrelated cell type.

Additional instructions:
- Use biological knowledge from single-cell RNA-seq studies and publication context.
- Return ONLY a JSON object in this format:

{
  "annotations": [
    {"cell_type": "...", "collection_doi": "...", "category": "..."},
    ...
  ]
}

Each cell type must be assigned exactly one category.
"""

In [ ]:
client = OpenAI()

# Prepare user message
user_message = {"cell_entries": cell_entries}

# Call GPT-4o
response = client.responses.create(
    input=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": json.dumps(user_message)}
    ]
)

# Extract output text
content = response.output_text
# If this fails, use: output_text = response.output[0].content[0].text

# Load JSON
try:
    results = json.loads(content)
except json.JSONDecodeError:
    print("Error parsing JSON from model output:")
    print(content)
    raise

annotations = results["annotations"]

# Convert back to DataFrame
df_annotations = pd.DataFrame(annotations)

df_annotations

In [ ]:
# Merge counts from original file
df_final = df_annotations.merge(df_summary[["cell_type", "count", "collection_doi"]],
                                on=['cell_type', 'collection_doi'],
                                how="left")

# Save final TSV
df_final.to_csv(os.path.join(input_dir, "celltype_category_assignments.tsv"),
                      sep="\t", index=False)

print("Saved: celltype_category_assignments.tsv")

In [ ]:
df_final